import necessary libraries
 ln1:pandas=data manipulation & analysis
 ln2:chembl-database(db)
    :new_client=object that allows you query the db via the API(fetch data)
          -Alternative:from chembl_webresource_client.new_client import new_client as chembl(clearer naming)

In [1]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

Target search for coronavirus
 ln1:target=usually a protein(in chembl)
 ln2:target_query=sends a query to chembl to find the targets related to "[protein]"(API call)
             =returns a list of dictonaries not a table yet
 ln3:pd.DataFrame=converts the query results into a df
             -Alternative:targets = pd.DataFrame(list(target_query))-forces evaluation
 ln4:targets=displays the table(esp in notebooks)


In [2]:
target = new_client.target
target_query = target.search('coronavirus')
targets = pd.DataFrame.from_dict(target_query)
targets

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Coronavirus,Coronavirus,17.0,False,CHEMBL613732,[],ORGANISM,11119
1,[],Feline coronavirus,Feline coronavirus,15.0,False,CHEMBL612744,[],ORGANISM,12663
2,[],Murine coronavirus,Murine coronavirus,15.0,False,CHEMBL5209664,[],ORGANISM,694005
3,[],Canine coronavirus,Canine coronavirus,15.0,False,CHEMBL5291668,[],ORGANISM,11153
4,[],Bovine coronavirus,Bovine coronavirus,15.0,False,CHEMBL6066646,[],ORGANISM,11128
5,[],Human coronavirus 229E,Human coronavirus 229E,13.0,False,CHEMBL613837,[],ORGANISM,11137
6,[],Human coronavirus OC43,Human coronavirus OC43,13.0,False,CHEMBL5209665,[],ORGANISM,31631
7,[],Middle East respiratory syndrome-related coron...,Middle East respiratory syndrome-related coron...,10.0,False,CHEMBL4296578,[],ORGANISM,1335626
8,[],Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,5.0,False,CHEMBL3927,"[{'accession': 'P0C6U8', 'component_descriptio...",SINGLE PROTEIN,694009
9,[],Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1ab,4.0,False,CHEMBL5118,"[{'accession': 'P0C6X7', 'component_descriptio...",SINGLE PROTEIN,694009


Select and retrieve bioactivity data for SARS coronavirus 3C-like proteinase(9th entry)-to study it further
ln1:selected_target=stores the index 8 (9th row) from the target_chembl_id column
   alternative:selected_target = targets['target_chembl_id']-safer opt
ln2:displays the selected ID(notebooks)
*the ID is used later to fect biodata and get compunds targeting the protein
*selected_target = targets['target_chembl_id'].iloc[4]
*targets[targets['pref_name'].str.contains('Aromatase', case=False)]
*targets.reset_index(drop=True, inplace=True)

In [3]:
selected_target = targets.target_chembl_id[8]
selected_target

'CHEMBL3927'

Retrieve only bioactivity data for coronavirus 3C-like proteinase (CHEMBL3927) that are reported as IC values in nM (nanomolar) unit.
*IC50 → how much drug is needed to inhibit 50% of activity(Inhibitory concentration(IC))
*Ki → binding affinity
*EC50 → effective concentration

 ln1:gets the activity resource from chembl
    Activity-how strongly a compound interacts with a target protein
 ln2:activity.filter(target_chembl_id=selected_target)=only deals with the chosen(selected)target
    :filter(standard_type='IC50')= further filters it to only IC50 values
    *filter()
*activity.filter(
    target_chembl_id=selected_target,
    standard_type="IC50"
 )[:100]

 *df.head(3)=only shows the first 3 rows

In [4]:
activity = new_client.activity
res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")

In [5]:
df=pd.DataFrame.from_dict(res)

In [6]:
df.head(3)

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,NaN,1480935,[],CHEMBL829584,In vitro inhibitory concentration against SARS...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,uM,UO_0000065,None,7.2
1,None,NaN,1480936,[],CHEMBL829584,In vitro inhibitory concentration against SARS...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,uM,UO_0000065,None,9.4
2,None,NaN,1481061,[],CHEMBL830868,In vitro inhibitory concentration against SARS...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,uM,UO_0000065,None,13.5


In [7]:
df.standard_type.unique()
#check if they are any odd ones

<StringArray>
['IC50']
Length: 1, dtype: str

In [8]:
#save the resulting bioactivity data to a CSV file (bioactivity_data.csv)not including the index(row numbers)-cleaner
#Comma Separated Values(CSV)
df.to_csv('bioactivity_data.csv',index=False)

df.to_csv('data/bioactivity_data.csv', index=False)-make sure the directory exists first
df.to_csv('bioactivity_data_2026-03-23.csv', index=False)-timestamp
df.to_csv('bioactivity_data.csv.gz', index=False)-commpressed for large datasets
df.to_csv('bioactivity_data.csv', index=False, encoding='utf-8')?????????????

In [9]:
import os
os.listdir('data')
#!ls -l data/ =timestamp

['bioactivity_data.csv', 'bioactivity_preprocessed_data.csv']

In [10]:
#handling missing data if any-those with no standard value-NaN=not a number
#df2 = df.dropna(subset=['standard_value'])
df2=df[df.standard_value.notna()]
df2

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,NaN,1480935,[],CHEMBL829584,In vitro inhibitory concentration against SARS...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,uM,UO_0000065,None,7.2
1,None,NaN,1480936,[],CHEMBL829584,In vitro inhibitory concentration against SARS...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,uM,UO_0000065,None,9.4
2,None,NaN,1481061,[],CHEMBL830868,In vitro inhibitory concentration against SARS...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,uM,UO_0000065,None,13.5
3,None,NaN,1481065,[],CHEMBL829584,In vitro inhibitory concentration against SARS...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,uM,UO_0000065,None,13.11
4,None,NaN,1481066,[],CHEMBL829584,In vitro inhibitory concentration against SARS...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,uM,UO_0000065,None,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256,"{'action_type': 'INHIBITOR', 'description': 'N...",NaN,25992021,"[{'comments': None, 'relation': '=', 'result_f...",CHEMBL5550822,Inhibition of SARS-CoV-1 main protease using D...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,nM,UO_0000065,None,52.0
257,"{'action_type': 'INHIBITOR', 'description': 'N...",NaN,25992022,"[{'comments': None, 'relation': '=', 'result_f...",CHEMBL5550822,Inhibition of SARS-CoV-1 main protease using D...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,nM,UO_0000065,None,30.0
258,"{'action_type': 'INHIBITOR', 'description': 'N...",NaN,25992023,"[{'comments': None, 'relation': '=', 'result_f...",CHEMBL5550822,Inhibition of SARS-CoV-1 main protease using D...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,nM,UO_0000065,None,14.0
259,"{'action_type': 'INHIBITOR', 'description': 'N...",NaN,25992024,"[{'comments': None, 'relation': '=', 'result_f...",CHEMBL5550822,Inhibition of SARS-CoV-1 main protease using D...,B,None,None,BAO_0000190,...,Severe acute respiratory syndrome-related coro...,Replicase polyprotein 1a,694009,None,None,IC50,nM,UO_0000065,None,48.0


Data pre-processing of the bioactivity data-labelling compounds as active inactive or intermediate
so,compounds<1000nM=Active
,compounds>10000nM=Inactive
,compounds btwn 1000nM and 10,000nM=Intermediate
i=one ic50 value at a time
float(i)=converts the value to a number
import numpy as np

conditions = [
    df2['standard_value'].astype(float) >= 10000,
    df2['standard_value'].astype(float) <= 1000
]

choices = ["inactive", "active"]

df2['bioactivity_class'] = np.select(conditions, choices, default="intermediate")
*Loops = slow for large datasets

In [11]:
bioactivity_class = []
for i in df2.standard_value:
  if float(i) >= 10000:
     bioactivity_class.append("inactive")
  elif float(i) <= 1000:
     bioactivity_class.append("active")
  else:
     bioactivity_class.append("intermediate")
     

iterate the molecule_chembl_id to a list
mol_cid = df2['molecule_chembl_id'].tolist()

In [12]:
mol_cid = []
for i in df2.molecule_chembl_id:
  mol_cid.append(i)
     

SMILES = Simplified Molecular Input Line Entry System

In [13]:
canonical_smiles = []
for i in df2.canonical_smiles:
  canonical_smiles.append(i)

In [14]:
standard_value = []
for i in df2.standard_value:
  standard_value.append(i)
  

In [15]:
data_tuples = list(zip(mol_cid, canonical_smiles, bioactivity_class, standard_value))
df3 = pd.DataFrame( data_tuples,  columns=['molecule_chembl_id', 'canonical_smiles', 'bioactivity_class', 'standard_value'])
     

In [16]:
df3

,molecule_chembl_id,canonical_smiles,bioactivity_class,standard_value
0,CHEMBL187579,Cc1noc(C)c1CN1C(=O)C(=O)c2cc(C#N)ccc21,intermediate,7200.0
1,CHEMBL188487,O=C1C(=O)N(Cc2ccc(F)cc2Cl)c2ccc(I)cc21,intermediate,9400.0
2,CHEMBL185698,O=C1C(=O)N(CC2COc3ccccc3O2)c2ccc(I)cc21,inactive,13500.0
3,CHEMBL426082,O=C1C(=O)N(Cc2cc3ccccc3s2)c2ccccc21,inactive,13110.0
4,CHEMBL187717,O=C1C(=O)N(Cc2cc3ccccc3s2)c2c1cccc2[N+](=O)[O-],intermediate,2000.0
...,...,...,...,...
254,CHEMBL5595277,CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CC...,active,52.0
255,CHEMBL5570210,CC(C)c1ccc(F)c2[nH]c(C(=O)N3C[C@H]4[C@@H]([C@H...,active,30.0
256,CHEMBL5565685,CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CC...,active,14.0
257,CHEMBL5565858,CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CC...,active,48.0


selection = ['molecule_chembl_id', 'canonical_smiles', 'standard_value']
df3 = df2[selection]
df3
    

In [17]:
pd.concat([df3,pd.Series(bioactivity_class)], axis=1)

,molecule_chembl_id,canonical_smiles,bioactivity_class,standard_value,0
0,CHEMBL187579,Cc1noc(C)c1CN1C(=O)C(=O)c2cc(C#N)ccc21,intermediate,7200.0,intermediate
1,CHEMBL188487,O=C1C(=O)N(Cc2ccc(F)cc2Cl)c2ccc(I)cc21,intermediate,9400.0,intermediate
2,CHEMBL185698,O=C1C(=O)N(CC2COc3ccccc3O2)c2ccc(I)cc21,inactive,13500.0,inactive
3,CHEMBL426082,O=C1C(=O)N(Cc2cc3ccccc3s2)c2ccccc21,inactive,13110.0,inactive
4,CHEMBL187717,O=C1C(=O)N(Cc2cc3ccccc3s2)c2c1cccc2[N+](=O)[O-],intermediate,2000.0,intermediate
...,...,...,...,...,...
254,CHEMBL5595277,CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CC...,active,52.0,active
255,CHEMBL5570210,CC(C)c1ccc(F)c2[nH]c(C(=O)N3C[C@H]4[C@@H]([C@H...,active,30.0,active
256,CHEMBL5565685,CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CC...,active,14.0,active
257,CHEMBL5565858,CC1(C)[C@@H]2[C@@H](C(=O)N[C@H](C=O)C[C@@H]3CC...,active,48.0,active


In [18]:
df3.to_csv('data/bioactivity_preprocessed_data.csv', index=False)
#best practice=v1 label(use versions)

In [19]:
!ls -l

total 544
-rw-r--r--  1 annngambi  staff  152799 Mar 26 21:53 bioactivity_data.csv
drwxr-xr-x  4 annngambi  staff     128 Mar 26 20:18 data
drwxr-xr-x  8 annngambi  staff     256 Mar 23 16:33 mienv
-rw-r--r--  1 annngambi  staff   59389 Mar 26 21:52 n1data_collection.ipynb
-rw-r--r--  1 annngambi  staff   51310 Mar 26 21:35 n2exploratory_data_analysis.ipynb
-rw-r--r--  1 annngambi  staff    3365 Mar 26 21:51 n3chemical_space_analysis.ipynb
-rw-r--r--  1 annngambi  staff    1866 Mar 23 16:46 requirements.txt
